# OpenAI Agents SDK — foundations

**Week 11 · Session 1**

Last week you built an agent by hand: a `while` loop, a tool schema, a dispatch dict,
a `messages` list, a step budget, a trace printer. About forty lines.

This week you get the same thing, packaged. Almost everything maps one-to-one:

| You wrote this in Week 10 | The Agents SDK gives you |
|---|---|
| the `run_agent()` while loop | `Runner.run()` |
| `TOOLS` schema + `DISPATCH` dict | `@function_tool` |
| `max_steps` | `max_turns` |
| the `messages` list / `Conversation` class | `Session` |
| your `trace()` printer | built-in tracing |
| the MCP → OpenAI `to_openai()` bridge | `mcp_servers=[...]` |

**Today we build a travel concierge**, one capability at a time. Tomorrow it gets
specialist agents, guardrails, a human approval step for bookings — and a voice.

> Nothing here is magic. When the SDK surprises you, reach for last week's mental
> model: messages, tool calls, a loop.

In [1]:
# %pip install -q "openai-agents==0.22.3" "mcp>=1.19,<2" python-dotenv

In [2]:
import os, sys, json, getpass
from pathlib import Path
from datetime import date, datetime

# Keys come from week11/.env if present; anything missing is prompted for.
try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True) or Path.cwd().parent / ".env")
except ImportError:
    pass
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")

from agents import (Agent, Runner, function_tool, ModelSettings, ItemHelpers,
                    RunContextWrapper, WebSearchTool, trace)

# httpx2 2.12+ decompresses brotli with output_buffer_limit=... as a keyword.
# Anaconda's brotli 1.x exposes process() as a C function that rejects kwargs
# ("TypeError: process() takes no keyword arguments"). Wrap it so OpenAI
# Responses (which often arrive Content-Encoding: br) can be decoded.
try:
    import brotli as _brotli
    from httpx2._decoders import BrotliDecoder as _BrotliDecoder
    _probe = _brotli.Decompressor()
    try:
        _probe.process(b"", output_buffer_limit=1)
    except TypeError:
        _brotli_init = _BrotliDecoder.__init__
        def _brotli_init_compat(self, *args, **kwargs):
            _brotli_init(self, *args, **kwargs)
            _impl = self._decompress
            self._decompress = lambda data, output_buffer_limit=None, **_k: _impl(data)
        _BrotliDecoder.__init__ = _brotli_init_compat
except Exception:
    pass

# The SDK's own default model changes with every release (right now: gpt-5.6-luna).
# We pin one explicitly: fast for live demos, cheap, and the same model as Week 10.
MODEL = "gpt-4.1-mini"
print("ready")

ready


A helper we'll use all session. It prints what the agent *did*, in last week's
vocabulary — so you can see the loop the SDK is running for you.

In [3]:
def show_run(result, width=110):
    """Print a run as ACTION / OBSERVATION / ANSWER, plus what it cost."""
    for item in result.new_items:
        kind = type(item).__name__
        raw = getattr(item, "raw_item", None)
        if kind == "ToolCallItem":
            name = getattr(raw, "name", None) or getattr(raw, "type", "tool")
            args = getattr(raw, "arguments", "") or ""
            print(f"  ACTION       {name}({args[:width]})")
        elif kind == "ToolCallOutputItem":
            print(f"  OBSERVATION  {str(item.output)[:width]}")
        elif kind == "HandoffOutputItem":
            print(f"  HANDOFF      {item.source_agent.name} -> {item.target_agent.name}")
        elif kind == "MessageOutputItem":
            print(f"  ANSWER       {ItemHelpers.text_message_output(item)[:width]}")
    u = result.context_wrapper.usage
    print(f"\n  {u.requests} model call(s) · {u.input_tokens} in + {u.output_tokens} out tokens"
          f" · last agent: {result.last_agent.name}")

---
# Part 1 — Your first agent

Two objects:

- **`Agent`** — *what* it is: instructions, a model, and (soon) tools.
- **`Runner`** — *how* it runs: the loop. It calls the model, runs tools, feeds results
  back, and stops when the model produces a final answer.

In [4]:
concierge = Agent(
    name="Travel Concierge",
    instructions="You are a friendly, concise travel concierge for travellers from India.",
    model=MODEL,
)

result = await Runner.run(concierge, "Suggest one weekend getaway from Bangalore, in two lines.")
print(result.final_output)

Head to Coorg, just a 5-6 hour drive from Bangalore, for lush coffee plantations and serene waterfalls. Perfect for a relaxing, nature-filled weekend escape!


> **`await` in a notebook.** Jupyter already runs an event loop, so `await Runner.run(...)`
> works directly in a cell. In a plain `.py` script use `Runner.run_sync(agent, "...")`.

What actually happened? The result object keeps the whole run:

In [5]:
show_run(result)
print("\nitems produced:", [type(i).__name__ for i in result.new_items])
print("raw model responses:", len(result.raw_responses))

  ANSWER       Head to Coorg, just a 5-6 hour drive from Bangalore, for lush coffee plantations and serene waterfalls. Perfec

  1 model call(s) · 35 in + 35 out tokens · last agent: Travel Concierge

items produced: ['MessageOutputItem']
raw model responses: 1


One model call, one message, no tools. That is **pattern 1 — the basic responder** from
last week's ladder, wrapped in a nicer API. Nothing agentic has happened yet.

---
# Part 2 — Instructions are the system prompt

Remember the six blocks from last week? They go straight into `instructions`.

In [6]:
CONCIERGE_RULES = """You are a travel concierge for travellers flying from India.

TOOL POLICY
- Always look up flights and hotels with your tools before quoting a price.
  Never invent a flight number, hotel name or price.
- Check visa rules for any international trip.

CONSTRAINTS
- Prices are in INR unless the traveller asks otherwise.
- You can SEARCH but you cannot BOOK. If asked to book, say booking is not available yet.

STOPPING CONDITION
- Stop once you have one recommended option per thing asked. Do not list everything.

OUTPUT
- Recommendation first, then at most two alternatives, then prices.

FAILURE
- If a tool fails twice, say what you tried. Never guess."""

### Dynamic instructions — fixing "the wall" from Week 10

Last week the model couldn't tell you today's date. Instead of adding a tool for that,
`instructions` can be a **function**. It runs on every turn, so the prompt is always
current.

In [7]:
def concierge_instructions(ctx: RunContextWrapper, agent: Agent) -> str:
    today = datetime.now().strftime("%A, %d %B %Y")
    return f"Today is {today}.\n\n{CONCIERGE_RULES}"


concierge = Agent(
    name="Travel Concierge",
    instructions=concierge_instructions,
    model=MODEL,
    model_settings=ModelSettings(temperature=0.3),   # steadier answers for a service agent
)

result = await Runner.run(concierge, "My flight is on 2 October 2026. What day of the week is that, "
                                    "and how many days do I have to plan?")
print(result.final_output)

2 October 2026 falls on a Friday. 

Since today is Saturday, 19 September 2026, you have 13 days left to plan your trip.


No tool call — the date simply arrived in the prompt. **Put what you already know into
the instructions. Use a tool for what you have to go and fetch.**

> **Try this:** ask *"How many days until Diwali?"* It gets *today* right, but it has to
> *guess* Diwali's date, and it may guess wrong. The prompt fixed the fact you supplied.
> It did nothing for the one you didn't. That's what tools are for.

---
# Part 3 — Structured output

In Week 10 the `OUTPUT CONTRACT` block *asked* the model for a format. Here we *enforce*
one. Give the agent a Pydantic model as `output_type`, and `final_output` comes back as a
typed Python object — validated, with no parsing and no regex.

In [8]:
from pydantic import BaseModel, Field


class TripIdea(BaseModel):
    destination: str
    best_month: str
    nights: int = Field(ge=1, le=14)
    three_things_to_do: list[str]
    rough_budget_inr: int = Field(description="Total per person, including flights")


planner = Agent(
    name="Trip Planner",
    instructions="Suggest exactly one trip that fits the request.",
    model=MODEL,
    output_type=TripIdea,
)

result = await Runner.run(planner, "A relaxed 3-night beach trip from Bangalore under 25,000 INR.")
idea = result.final_output

print(type(idea).__name__)
print(idea.model_dump_json(indent=2))

TripIdea
{
  "destination": "Pondicherry",
  "best_month": "January",
  "nights": 3,
  "three_things_to_do": [
    "Relax on Promenade Beach",
    "Explore French Quarter",
    "Visit Auroville and Matrimandir"
  ],
  "rough_budget_inr": 23000
}


In [9]:
# It's an object, so ordinary code can use it directly.
print(f"Book {idea.nights} nights in {idea.destination} around {idea.best_month}.")
print(f"Budget per night, roughly: INR {idea.rough_budget_inr // idea.nights:,}")

Book 3 nights in Pondicherry around January.
Budget per night, roughly: INR 7,666


This is the difference between *an answer* and *data*. Anything downstream — a UI, a
database, another agent — can consume `TripIdea` without guessing.

---
# Part 4 — Streaming

Nobody wants to stare at a spinner. `run_streamed` yields tokens as they arrive, plus
higher-level events: which tool was called, when the output came back, when the agent
changed.

In [10]:
from openai.types.responses import ResponseTextDeltaEvent

stream = Runner.run_streamed(concierge, "Give me three packing tips for Goa in October.")
async for event in stream.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)
print()

Here

 are

 three

 packing

 tips

...

 to

 Goa

 in

 October

!

---
# Part 5 — Tools

## Our (pretend) travel inventory

A small, fixed dataset so everyone's results match. **Demo data only** — the visa rules
below are illustrative, not current policy.

In [11]:
FLIGHTS = [
    {"id": "6E-512",  "from": "BLR", "to": "GOI", "date": "2026-10-02", "depart": "07:10", "arrive": "08:20", "airline": "IndiGo",             "price_inr": 4180},
    {"id": "AI-887",  "from": "BLR", "to": "GOI", "date": "2026-10-02", "depart": "13:45", "arrive": "15:00", "airline": "Air India",          "price_inr": 5620},
    {"id": "QP-1375", "from": "BLR", "to": "GOI", "date": "2026-10-02", "depart": "18:30", "arrive": "19:35", "airline": "Akasa Air",          "price_inr": 3890},
    {"id": "6E-513",  "from": "GOI", "to": "BLR", "date": "2026-10-05", "depart": "09:00", "arrive": "10:10", "airline": "IndiGo",             "price_inr": 4350},
    {"id": "QP-1376", "from": "GOI", "to": "BLR", "date": "2026-10-05", "depart": "20:10", "arrive": "21:15", "airline": "Akasa Air",          "price_inr": 3990},
    {"id": "EK-569",  "from": "BLR", "to": "DXB", "date": "2026-10-02", "depart": "04:30", "arrive": "07:05", "airline": "Emirates",           "price_inr": 21400},
    {"id": "6E-1485", "from": "BLR", "to": "DXB", "date": "2026-10-02", "depart": "22:05", "arrive": "00:40", "airline": "IndiGo",             "price_inr": 16800},
    {"id": "SQ-511",  "from": "BLR", "to": "SIN", "date": "2026-10-02", "depart": "23:55", "arrive": "06:55", "airline": "Singapore Airlines", "price_inr": 24900},
    {"id": "6E-1005", "from": "BLR", "to": "SIN", "date": "2026-10-02", "depart": "01:20", "arrive": "08:30", "airline": "IndiGo",             "price_inr": 18200},
]

HOTELS = {
    "goa": [
        {"id": "H-GOA-1", "name": "Sea Breeze Candolim",      "area": "Candolim, 200 m from the beach", "price_inr": 5200, "rating": 4.3},
        {"id": "H-GOA-2", "name": "Fontainhas Heritage Stay", "area": "Panjim old quarter",             "price_inr": 3800, "rating": 4.6},
        {"id": "H-GOA-3", "name": "Anjuna Cliff Resort",      "area": "Anjuna, cliff-top",              "price_inr": 9800, "rating": 4.5},
        {"id": "H-GOA-4", "name": "Palolem Palms Cottages",   "area": "Palolem beach, South Goa",       "price_inr": 4400, "rating": 4.2},
    ],
    "dubai": [
        {"id": "H-DXB-1", "name": "Marina View Hotel", "area": "Dubai Marina", "price_inr": 11200, "rating": 4.4},
        {"id": "H-DXB-2", "name": "Deira Budget Inn",  "area": "Deira",        "price_inr": 5600,  "rating": 3.9},
    ],
    "singapore": [
        {"id": "H-SIN-1", "name": "Bugis Boutique",    "area": "Bugis",        "price_inr": 12800, "rating": 4.3},
        {"id": "H-SIN-2", "name": "Little India Lodge", "area": "Little India", "price_inr": 6900,  "rating": 4.0},
    ],
}

# DEMO DATA -- illustrative only. Real visa rules change; always check the embassy.
VISA_RULES = {
    ("india", "goa"):       "Domestic travel. No visa. Carry a government photo ID.",
    ("india", "dubai"):     "Visa required for most Indian passport holders (e-visa). "
                            "Some holders of valid US/UK/EU visas may get visa-on-arrival.",
    ("india", "singapore"): "Visa required. Apply online through an authorised agent, "
                            "typically 3-5 working days.",
}
print(f"{len(FLIGHTS)} flights · {sum(len(v) for v in HOTELS.values())} hotels · {len(VISA_RULES)} visa rules")

9 flights · 8 hotels · 3 visa rules


## `@function_tool` — the schema you hand-wrote last week, generated

Decorate an ordinary Python function. The SDK builds the JSON schema from your **type
hints**, and the description from your **docstring**, including each argument's `Args:`
line.

In [12]:
@function_tool
def search_flights(origin: str, destination: str, date: str) -> str:
    """Search available flights between two airports on a given date.

    Args:
        origin: IATA code of the departure airport, e.g. "BLR" for Bangalore.
        destination: IATA code of the arrival airport, e.g. "GOI" for Goa.
        date: Travel date in YYYY-MM-DD format.
    """
    hits = [f for f in FLIGHTS
            if f["from"] == origin.upper() and f["to"] == destination.upper() and f["date"] == date]
    if not hits:
        routes = sorted({f"{f['from']}->{f['to']} on {f['date']}" for f in FLIGHTS})
        return f"No flights {origin}->{destination} on {date}. Routes we have: {', '.join(routes)}"
    return json.dumps(sorted(hits, key=lambda f: f["price_inr"]))


@function_tool
def search_hotels(city: str, max_price_inr: int = 100000) -> str:
    """Find hotels in a city, cheapest first, optionally under a nightly budget.

    Args:
        city: City name, e.g. "Goa".
        max_price_inr: Maximum price per night in INR.
    """
    options = HOTELS.get(city.strip().lower())
    if options is None:
        return f"No hotels for '{city}'. Cities we cover: {', '.join(c.title() for c in HOTELS)}"
    fits = [h for h in options if h["price_inr"] <= max_price_inr]
    if not fits:
        return f"Nothing in {city} under INR {max_price_inr}. Cheapest is INR {min(h['price_inr'] for h in options)}."
    return json.dumps(sorted(fits, key=lambda h: h["price_inr"]))


@function_tool
def check_visa(passport_country: str, destination: str) -> str:
    """Look up visa requirements for a passport holder travelling to a destination.

    Args:
        passport_country: Country that issued the passport, e.g. "India".
        destination: Destination city, e.g. "Dubai".
    """
    rule = VISA_RULES.get((passport_country.strip().lower(), destination.strip().lower()))
    return rule or f"No rule on file for {passport_country} -> {destination}. Advise checking the embassy."

In [13]:
# Compare this with the SEARCH_TOOL dict you typed out by hand in Week 10.
print("name       :", search_flights.name)
print("description:", search_flights.description)
print(json.dumps(search_flights.params_json_schema, indent=2))

name       : search_flights
description: Search available flights between two airports on a given date.
{
  "properties": {
    "origin": {
      "description": "IATA code of the departure airport, e.g. \"BLR\" for Bangalore.",
      "title": "Origin",
      "type": "string"
    },
    "destination": {
      "description": "IATA code of the arrival airport, e.g. \"GOI\" for Goa.",
      "title": "Destination",
      "type": "string"
    },
    "date": {
      "description": "Travel date in YYYY-MM-DD format.",
      "title": "Date",
      "type": "string"
    }
  },
  "required": [
    "origin",
    "destination",
    "date"
  ],
  "title": "search_flights_args",
  "type": "object",
  "additionalProperties": false
}


Every `Args:` line became a parameter description. **Your docstring is now prompt text** —
exactly last week's "tool descriptions are prompts" lesson, except now the docstring
*is* the description. A lazy docstring is a lazy prompt.

In [14]:
concierge = Agent(
    name="Travel Concierge",
    instructions=concierge_instructions,
    model=MODEL,
    model_settings=ModelSettings(temperature=0.3),
    tools=[search_flights, search_hotels, check_visa],
)

result = await Runner.run(
    concierge,
    "I'm flying Bangalore to Goa on 2 October 2026. Cheapest flight, and a hotel near the beach under 6000 a night.",
)
show_run(result)

  ACTION       search_flights({"origin":"BLR","destination":"GOI","date":"2026-10-02"})
  ACTION       search_hotels({"city":"Goa","max_price_inr":6000})
  OBSERVATION  [{"id": "QP-1375", "from": "BLR", "to": "GOI", "date": "2026-10-02", "depart": "18:30", "arrive": "19:35", "ai
  OBSERVATION  [{"id": "H-GOA-2", "name": "Fontainhas Heritage Stay", "area": "Panjim old quarter", "price_inr": 3800, "ratin
  ANSWER       For your flight from Bangalore to Goa on 2 October 2026, the cheapest option is with Akasa Air, departing at 1

  2 model call(s) · 1303 in + 217 out tokens · last agent: Travel Concierge


Read the trace. The model chose **which** tools, with **what** arguments, and in what
**order** — and it probably called two tools in the same step. That is your Week 10
loop: `Runner` did the dispatch, appended the observations, and called the model again.

### When a tool fails

In Week 10 we said *the agent reads your error message*. Watch what the SDK tells the
model when a tool raises:

In [15]:
@function_tool
def live_hotel_price(hotel_id: str) -> str:
    """Get the live price for a hotel by id."""
    raise TimeoutError("pricing API did not respond in 5s")


probe = Agent(name="Probe", instructions="Use the tool, then report the price.", model=MODEL,
              tools=[live_hotel_price])
result = await Runner.run(probe, "What's the live price for H-GOA-1?")
for item in result.new_items:
    if type(item).__name__ == "ToolCallOutputItem":
        print("The model was told:", repr(item.output))
print("\nIt answered:", result.final_output)

The model was told: 'An error occurred while running the tool. Please try again. Error: pricing API did not respond in 5s'

It answered: I encountered an issue while trying to get the live price for the hotel with ID H-GOA-1. Could you please try again later?


Notice **"Please try again."** That's the SDK's default — and count how many times the
tool was called above. If it's more than once, the model took that advice and retried a
dead API. Against a flaky service that's a retry loop you're paying for. You control this: pass `failure_error_function` to
`@function_tool`, or — better — catch the error inside your tool and return a message
that says what to do next. Just as you did in Week 10.

---
# Part 6 — Hosted tools

Everything so far runs **your** Python. Hosted tools run on **OpenAI's** side: web
search, file search over your documents, a code interpreter. No Tavily key, no code to
write — and no way to see inside them.

In [16]:
scout = Agent(
    name="Destination Scout",
    instructions="Answer from the live web. Keep it to three lines and cite one source URL.",
    model=MODEL,
    tools=[WebSearchTool()],
)

result = await Runner.run(scout, "What's the weather like in Goa this week, and is it still monsoon?")
show_run(result)
print("\n" + result.final_output)

  ACTION       web_search_call()
  ANSWER       This week in Goa, expect partly sunny to mostly cloudy conditions with temperatures ranging from 77°F (25°C) t

  1 model call(s) · 8174 in + 400 out tokens · last agent: Destination Scout

This week in Goa, expect partly sunny to mostly cloudy conditions with temperatures ranging from 77°F (25°C) to 91°F (33°C). There's a chance of light rain or thunderstorms, especially in the afternoons and evenings. The monsoon season in Goa typically lasts from June to September, so it has ended by mid-September. ([pageweather.com](https://pageweather.com/weather/goa/september?utm_source=openai))

## Weather for Panaji, India:
Current Conditions: Partly sunny, 86°F (30°C)

Daily Forecast:
* Saturday, September 19: Low: 77°F (25°C), High: 91°F (33°C), Description: Sun and some clouds
* Sunday, September 20: Low: 78°F (26°C), High: 87°F (31°C), Description: Partly sunny
* Monday, September 21: Low: 79°F (26°C), High: 85°F (30°C), Description: High thin

Look at the input tokens on that one run. The search results were pulled into the
context — Week 10's context budget, arriving from a tool you can't even see into.

| | Function tool | Hosted tool |
|---|---|---|
| Who runs it | your code | OpenAI |
| You can see / change it | yes | no |
| Needs | nothing extra | a Responses-API model |
| Cost | your infra | billed per call |

---
# Part 7 — Agents as tools

An agent can be a tool for another agent. The concierge stays in charge; the specialist
answers one question and hands the answer back.

In [17]:
visa_expert = Agent(
    name="Visa Expert",
    instructions=("You answer visa questions for Indian passport holders. Always use check_visa. "
                  "Two lines max. Always end with: 'Confirm with the embassy before booking.'"),
    model=MODEL,
    tools=[check_visa],
)

concierge = Agent(
    name="Travel Concierge",
    instructions=concierge_instructions,
    model=MODEL,
    tools=[
        search_flights,
        search_hotels,
        visa_expert.as_tool(
            tool_name="ask_visa_expert",
            tool_description="Ask the visa specialist about entry requirements for a destination.",
        ),
    ],
)

result = await Runner.run(concierge, "Thinking of Dubai on 2 Oct 2026 from Bangalore. Do I need a visa, and what's the cheapest flight?")
show_run(result)

  ACTION       ask_visa_expert({"input":"Visa requirements for Indian citizens traveling to Dubai in October 2026"})
  ACTION       search_flights({"origin":"BLR","destination":"DXB","date":"2026-10-02"})
  OBSERVATION  Indian citizens generally need an e-visa to visit Dubai; some with valid US/UK/EU visas may get visa-on-arriva
  OBSERVATION  [{"id": "6E-1485", "from": "BLR", "to": "DXB", "date": "2026-10-02", "depart": "22:05", "arrive": "00:40", "ai
  ANSWER       For your trip from Bangalore to Dubai on 2 Oct 2026, Indian citizens generally need an e-visa to enter Dubai. 

  4 model call(s) · 1406 in + 268 out tokens · last agent: Travel Concierge


The concierge called `ask_visa_expert` like any other tool. Inside it, a *whole second
agent* ran its own loop, with its own tool — which is why the model-call count is higher
than the number of steps you can see. Every specialist's loop is on your bill.

**Hold this thought for tomorrow.** `as_tool` means *the manager keeps control*. The
alternative, a **handoff**, means *control transfers* to the specialist. Choosing between
them is the main design decision in multi-agent systems.

---
# Part 8 — MCP: plug in a server, unchanged

Last week you built MCP servers with FastMCP, then wrote a `to_openai()` bridge and a
dispatch step to use them from your loop. The SDK does both for you.

First, a tiny travel-facts server — the same FastMCP you already know:

In [18]:
%%writefile travel_mcp_server.py
"""Travel facts MCP server (FastMCP, stdio). Written by the Week 11 notebook."""
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("travel-facts")

RATES_TO_INR = {"INR": 1.0, "USD": 88.2, "EUR": 102.9, "GBP": 118.4, "AED": 24.0, "SGD": 68.6}
FACTS = {
    "goa":       {"country": "India",     "currency": "INR", "timezone": "UTC+5:30", "plug": "Type C/D", "emergency": "112"},
    "dubai":     {"country": "UAE",       "currency": "AED", "timezone": "UTC+4",    "plug": "Type G",   "emergency": "999"},
    "singapore": {"country": "Singapore", "currency": "SGD", "timezone": "UTC+8",    "plug": "Type G",   "emergency": "999"},
}


@mcp.tool()
def convert_currency(amount: float, from_currency: str, to_currency: str) -> str:
    """Convert an amount between currencies at indicative rates.

    Args:
        amount: The amount to convert.
        from_currency: ISO code, e.g. "INR".
        to_currency: ISO code, e.g. "SGD".
    """
    f, t = from_currency.upper(), to_currency.upper()
    if f not in RATES_TO_INR or t not in RATES_TO_INR:
        return f"Unsupported currency. Supported: {', '.join(RATES_TO_INR)}"
    return f"{amount:,.2f} {f} = {amount * RATES_TO_INR[f] / RATES_TO_INR[t]:,.2f} {t} (indicative)"


@mcp.tool()
def destination_facts(city: str) -> str:
    """Practical facts for a destination: currency, timezone, plug type, emergency number.

    Args:
        city: City name, e.g. "Dubai".
    """
    f = FACTS.get(city.strip().lower())
    if not f:
        return f"No facts for '{city}'. Known: {', '.join(c.title() for c in FACTS)}"
    return "; ".join(f"{k}: {v}" for k, v in f.items())


if __name__ == "__main__":
    mcp.run(transport="stdio")

Writing travel_mcp_server.py


In [19]:
from agents.mcp import MCPServerStdio

SERVER = str(Path("travel_mcp_server.py").resolve())

async with MCPServerStdio(
    name="travel-facts",
    params={"command": sys.executable, "args": [SERVER]},
    cache_tools_list=True,
) as travel_facts:
    print("discovered over MCP:", [t.name for t in await travel_facts.list_tools()])

    concierge_mcp = Agent(
        name="Travel Concierge",
        instructions=concierge_instructions,
        model=MODEL,
        tools=[search_flights, search_hotels],
        mcp_servers=[travel_facts],        # <- that's the whole integration
    )
    result = await Runner.run(concierge_mcp,
        "I'm taking 50,000 INR to Singapore. How much is that in SGD, and what plug adapter do I need?")

show_run(result)

discovered over MCP: ['convert_currency', 'destination_facts']


  ACTION       convert_currency({"amount":50000,"from_currency":"INR","to_currency":"SGD"})
  ACTION       destination_facts({"city":"Singapore"})
  OBSERVATION  {'type': 'text', 'text': '50,000.00 INR = 728.86 SGD (indicative)'}
  OBSERVATION  {'type': 'text', 'text': 'country: Singapore; currency: SGD; timezone: UTC+8; plug: Type G; emergency: 999'}
  ANSWER       50,000 INR is approximately 728.86 SGD at indicative rates. For Singapore, you will need a Type G plug adapter

  2 model call(s) · 1104 in + 102 out tokens · last agent: Travel Concierge


Compare with last week's `mcp_client_with_llm` notebook. `list_tools()`, the
`to_openai()` conversion, `call_tool()` dispatch — all gone. **One line:**
`mcp_servers=[travel_facts]`. Function tools and MCP tools sit side by side, and the
model can't tell them apart.

Remote servers work the same way with `MCPServerStreamableHttp` — the transport you
used when we connected a hosted server last week.

---
# Part 9 — Memory: Sessions

Every `Runner.run` so far started from nothing. In Week 10 you fixed that with a
`Conversation` class holding a `messages` list. The SDK's version is a **Session**.

In [20]:
from agents import SQLiteSession

concierge = Agent(
    name="Travel Concierge",
    instructions=concierge_instructions,
    model=MODEL,
    model_settings=ModelSettings(temperature=0.3),
    tools=[search_flights, search_hotels, check_visa],
)

trip = SQLiteSession("asha-goa-trip")          # in-memory; add a db path to persist

for turn in ["Find me flights from Bangalore to Goa on 2 October 2026.",
             "Which of those leaves in the afternoon?",
             "Good. Now a hotel near the beach, under 6000 a night.",
             "Summarise my plan in three bullet points with prices."]:
    result = await Runner.run(concierge, turn, session=trip)
    print(f"YOU:   {turn}\nAGENT: {result.final_output}\n")

YOU:   Find me flights from Bangalore to Goa on 2 October 2026.
AGENT: The best flight option from Bangalore to Goa on 2 October 2026 is with Akasa Air, departing at 18:30 and arriving at 19:35, priced at INR 3890.

Alternatives include:
1. IndiGo flight departing at 07:10 and arriving at 08:20 for INR 4180.
2. Air India flight departing at 13:45 and arriving at 15:00 for INR 5620.

Would you like me to help with hotels or anything else?



YOU:   Which of those leaves in the afternoon?
AGENT: The Air India flight leaves in the afternoon at 13:45 and arrives in Goa at 15:00. The other two flights depart in the morning and evening.



YOU:   Good. Now a hotel near the beach, under 6000 a night.
AGENT: I recommend Palolem Palms Cottages near Palolem beach in South Goa, priced at INR 4400 per night. It offers a good balance of location and comfort near the beach.

Alternatives include:
1. Sea Breeze Candolim, about 200 meters from Candolim beach, priced at INR 5200 per night.
2. Fontainhas Heritage Stay in Panjim old quarter, priced at INR 3800 per night.

Would you like more details or help with anything else?



YOU:   Summarise my plan in three bullet points with prices.
AGENT: - Flight: Air India from Bangalore to Goa, departing at 13:45 on 2 October 2026, priced at INR 5620.
- Hotel: Palolem Palms Cottages near Palolem beach, South Goa, at INR 4400 per night.
- Alternative flight options: Akasa Air at INR 3890 (evening) and IndiGo at INR 4180 (morning). Alternative hotels: Sea Breeze Candolim at INR 5200 and Fontainhas Heritage Stay at INR 3800.



*"Which of those"*, *"Good"*, *"my plan"* — none of those work without memory. The
session stored every turn and replayed it into the next call.

In [21]:
items = await trip.get_items()
print(f"{len(items)} items stored in the session")
print("roles:", [i.get("role") or i.get("type") for i in items][:14], "...")

# A fresh session knows nothing. Same agent, same question.
fresh = await Runner.run(concierge, "Summarise my plan.", session=SQLiteSession("someone-else"))
print("\nfresh session ->", fresh.final_output[:160])

12 items stored in the session
roles: ['user', 'function_call', 'function_call_output', 'assistant', 'user', 'assistant', 'user', 'function_call', 'function_call_output', 'assistant', 'user', 'assistant'] ...



fresh session -> I don't have any details about your plan yet. Could you please provide the details or itinerary you want me to summarize?


`SQLiteSession("id", "trips.db")` persists to disk and survives restarts. There are
also Redis, SQLAlchemy and OpenAI-hosted sessions.

Remember Week 10's **context budget**: every stored item is re-sent on every turn. The
SDK ships `OpenAIResponsesCompactionSession`, which compacts old turns — the
summarise-and-drop fix you built by hand, as a drop-in.

---
# Part 10 — Context: data for your code, not for the model

Tools often need to know *who* is asking: a user id, their preferences, a database
handle. You could put that in the prompt and hope the model passes it back correctly.
Don't. Pass it as **context**.

In [22]:
from dataclasses import dataclass


@dataclass
class Traveller:
    name: str
    home_airport: str
    passport_country: str
    budget_per_night_inr: int
    loyalty_id: str              # sensitive: the model should never see this


@function_tool
def my_profile(ctx: RunContextWrapper[Traveller]) -> str:
    """Get the traveller's name, home airport, passport country and hotel budget."""
    t = ctx.context
    return (f"{t.name}; home airport {t.home_airport}; passport {t.passport_country}; "
            f"hotel budget INR {t.budget_per_night_inr}/night")


def personal_instructions(ctx: RunContextWrapper[Traveller], agent) -> str:
    return (f"Today is {date.today():%d %B %Y}. You are {ctx.context.name}'s concierge. "
            f"Call my_profile before planning.\n\n{CONCIERGE_RULES}")


personal = Agent[Traveller](
    name="Personal Concierge",
    instructions=personal_instructions,
    model=MODEL,
    tools=[my_profile, search_flights, search_hotels, check_visa],
)

asha = Traveller("Asha", "BLR", "India", 5000, loyalty_id="LOY-88213-SECRET")
result = await Runner.run(personal, "Plan my Goa trip on 2 October 2026.", context=asha)
show_run(result)

  ACTION       my_profile({})
  OBSERVATION  Asha; home airport BLR; passport India; hotel budget INR 5000/night
  ACTION       search_flights({"origin":"BLR","destination":"GOI","date":"2026-10-02"})
  ACTION       search_hotels({"city":"Goa","max_price_inr":5000})
  ACTION       check_visa({"passport_country":"India","destination":"Goa"})
  OBSERVATION  [{"id": "QP-1375", "from": "BLR", "to": "GOI", "date": "2026-10-02", "depart": "18:30", "arrive": "19:35", "ai
  OBSERVATION  [{"id": "H-GOA-2", "name": "Fontainhas Heritage Stay", "area": "Panjim old quarter", "price_inr": 3800, "ratin
  OBSERVATION  Domestic travel. No visa. Carry a government photo ID.
  ANSWER       For your Goa trip on 2 October 2026, here is the plan:

Flight:
- Best option: Akasa Air, departing from BLR a

  3 model call(s) · 1859 in + 289 out tokens · last agent: Personal Concierge


In [23]:
# Did the loyalty id ever reach the model? Check everything it was sent:
# the instructions, plus every message, tool call and tool output in the run.
sent_instructions = await personal.get_system_prompt(result.context_wrapper)
sent_conversation = json.dumps(result.to_input_list(), default=str)
print("loyalty id in instructions :", "LOY-88213" in sent_instructions)
print("loyalty id in conversation :", "LOY-88213" in sent_conversation)
print("but the tool could read it :", asha.loyalty_id)

loyalty id in instructions : False
loyalty id in conversation : False
but the tool could read it : LOY-88213-SECRET


**Context is never sent to the model.** It's a Python object that your tools and
instruction functions can read. The model only sees what a tool *returns*.

That makes context the right home for user ids, auth tokens, database connections and
feature flags. It's dependency injection for agents, and it's also a security boundary.

---
# Part 11 — Tracing

Last week you printed a trace by hand. The SDK records one for every run, automatically:
every model call, tool call and handoff, with timings. Open it here:
**https://platform.openai.com/traces**

In [24]:
with trace("Asha plans Goa") as t:
    trip = SQLiteSession("traced-trip")
    await Runner.run(personal, "Cheapest flight BLR to GOI on 2 October 2026?", context=asha, session=trip)
    await Runner.run(personal, "And a hotel under my budget.", context=asha, session=trip)

print("trace id:", t.trace_id)
print("open it: https://platform.openai.com/traces/trace?trace_id=" + t.trace_id)

trace id: trace_2209c09fd4bb4cbd8e3359769eae3690
open it: https://platform.openai.com/traces/trace?trace_id=trace_2209c09fd4bb4cbd8e3359769eae3690


Wrapping several runs in one `trace(...)` groups them into one workflow in the
dashboard — one trip, not two unrelated runs.

### Your Week 10 `trace()`, as a plugin

Traces go to OpenAI by default. You can also attach your own **processor** — to print
locally, or to send spans to Langfuse, Datadog, or your own logs:

In [25]:
from agents import TracingProcessor, add_trace_processor


class PrintSpans(TracingProcessor):
    """Prints each span as it finishes -- last week's trace(), as an SDK plugin."""
    def on_trace_start(self, t):
        print(f"┌─ trace: {t.name}")
    def on_trace_end(self, t):
        print(f"└─ end: {t.name}")
    def on_span_start(self, span):
        pass
    def on_span_end(self, span):
        data = span.span_data
        kind = type(data).__name__.replace("SpanData", "")
        label = getattr(data, "name", "") or ""
        ms = ""
        if span.started_at and span.ended_at:
            ms = f"{(datetime.fromisoformat(span.ended_at) - datetime.fromisoformat(span.started_at)).total_seconds() * 1000:.0f} ms"
        print(f"│  {kind:<10} {label:<22} {ms}")
    def shutdown(self):
        pass
    def force_flush(self):
        pass


add_trace_processor(PrintSpans())

with trace("Asha checks Dubai"):
    await Runner.run(personal, "Could I do Dubai instead on 2 October 2026? Visa and cheapest flight.", context=asha)

┌─ trace: Asha checks Dubai


│  Response                          944 ms
│  Function   my_profile             1 ms
│  Turn                              947 ms


│  Response                          1322 ms
│  Function   check_visa             1 ms
│  Function   search_flights         1 ms
│  Turn                              1326 ms


│  Response                          1589 ms
│  Turn                              1591 ms
│  Agent      Personal Concierge     3864 ms
│  Task       Agent workflow         3865 ms
└─ end: Asha checks Dubai


The processor stays registered for the rest of the notebook, so Part 12 prints its spans
too. (Re-running this cell adds a second copy — restart the kernel if you see double.)

Read it top to bottom: `Response` spans are model calls, `Function` spans are your
tools, and `Agent` wraps the whole run. When a run is slow or expensive, this is where
you find out which step did it.

---
# Part 12 — Putting it together

Everything from today, in one agent: dynamic instructions, function tools, a specialist
agent-as-tool, live web search, an MCP server, memory, context and tracing.

In [26]:
async with MCPServerStdio(name="travel-facts",
                          params={"command": sys.executable, "args": [SERVER]},
                          cache_tools_list=True) as travel_facts:

    concierge_v1 = Agent[Traveller](
        name="Travel Concierge v1",
        instructions=personal_instructions,
        model=MODEL,
        model_settings=ModelSettings(temperature=0.3),
        tools=[
            my_profile,
            search_flights,
            search_hotels,
            visa_expert.as_tool(tool_name="ask_visa_expert",
                                tool_description="Ask the visa specialist about entry requirements."),
            WebSearchTool(),
        ],
        mcp_servers=[travel_facts],
    )

    session = SQLiteSession("asha-v1")
    with trace("Concierge v1 demo"):
        for turn in ["I want 3 nights in Singapore from 2 October 2026. Flights, a hotel in my budget, "
                     "and do I need a visa?",
                     "What's that hotel's nightly price in SGD, and what plug do I pack?"]:
            result = await Runner.run(concierge_v1, turn, context=asha, session=session, max_turns=12)
            print(f"\nYOU: {turn}")
            show_run(result)
            print("\n" + result.final_output)

┌─ trace: Concierge v1 demo
│  MCPListTools                        2 ms


│  Response                          1082 ms
│  Function   my_profile             2 ms
│  Turn                              1087 ms
│  MCPListTools                        0 ms


│  Response                          1205 ms
│  Function   search_flights         1 ms
│  Turn                              1208 ms
│  MCPListTools                        0 ms


│  Response                          1195 ms
│  Function   search_hotels          1 ms
│  Turn                              1198 ms
│  MCPListTools                        0 ms


│  Response                          1598 ms


...

│  Response                          1464 ms
│  Turn                              1466 ms
│  Agent      Visa Expert            2621 ms
│  Task       Agent workflow         2621 ms
│  Function   ask_visa_expert        2622 ms
│  Turn                              4222 ms
│  MCPListTools                        0 ms


│  Response                          2291 ms
│  Turn                              2293 ms
│  Agent      Travel Concierge v1    10014 ms
│  Task       Agent workflow         10017 ms

YOU: I want 3 nights in Singapore from 2 October 2026. Flights, a hotel in my budget, and do I need a visa?
  ACTION       my_profile({})
  OBSERVATION  Asha; home airport BLR; passport India; hotel budget INR 5000/night
  ACTION       search_flights({"origin":"BLR","destination":"SIN","date":"2026-10-02"})
  OBSERVATION  [{"id": "6E-1005", "from": "BLR", "to": "SIN", "date": "2026-10-02", "depart": "01:20", "arrive": "08:30", "ai
  ACTION       search_hotels({"city":"Singapore","max_price_inr":5000})
  OBSERVATION  Nothing in Singapore under INR 5000. Cheapest is INR 6900.
  ACTION       ask_visa_expert({"input":"Visa requirements for Indian passport holders traveling to Singapore for tourism"})
  OBSERVATION  Indian passport holders require a visa for tourism to Singapore, which can be applied online via

│  Response                          1182 ms
│  Function   convert_currency       6 ms
│  Turn                              1191 ms
│  MCPListTools                        0 ms


│  Response                          1222 ms
│  Function   destination_facts      6 ms
│  Turn                              1230 ms
│  MCPListTools                        0 ms


│  Response                          1284 ms
│  Turn                              1288 ms
│  Agent      Travel Concierge v1    3713 ms
│  Task       Agent workflow         3714 ms

YOU: What's that hotel's nightly price in SGD, and what plug do I pack?
  ACTION       convert_currency({"amount":6900,"from_currency":"INR","to_currency":"SGD"})
  OBSERVATION  {'type': 'text', 'text': '6,900.00 INR = 100.58 SGD (indicative)'}
  ACTION       destination_facts({"city":"Singapore"})
  OBSERVATION  {'type': 'text', 'text': 'country: Singapore; currency: SGD; timezone: UTC+8; plug: Type G; emergency: 999'}
  ANSWER       The cheapest hotel's nightly price of INR 6,900 is approximately 100.58 SGD.

For Singapore, you should pack a

  3 model call(s) · 4385 in + 91 out tokens · last agent: Travel Concierge v1

The cheapest hotel's nightly price of INR 6,900 is approximately 100.58 SGD.

For Singapore, you should pack a Type G plug adapter. The local currency is SGD, and the emergency number is 99

Read the end of the first answer. The rules say *"you cannot BOOK"* — and it probably
still offered to book something. **Instructions are requests, not enforcement.**
Tomorrow's guardrails and approval steps are the enforcement.

`max_turns=12` is Week 10's `max_steps`. Leave it off and the default is 10. Hit the
limit and the SDK raises `MaxTurnsExceeded` rather than looping forever.

## What the concierge still can't do → tomorrow

- **Book anything.** Booking is a *write*, and a write needs a **human approval** step.
- **Refuse bad input.** Nothing stops someone pasting a passport number, or asking it
  to write their homework. That's what **guardrails** are for.
- **Specialise.** One agent with every tool gets worse as the tool list grows. Tomorrow:
  a triage agent that **hands off** to flight, hotel and visa specialists.
- **Talk.** It's text only. Tomorrow it gets a **voice**.

---
## Exercises

**1. Add a tool.** `search_trains(origin, destination, date)` with a small dataset. Does
the concierge start suggesting trains for Goa on its own? What in the instructions would
make it?

**2. Break a docstring.** Change `search_hotels`'s docstring to just `"Hotels."` and
re-run the Part 5 question. Compare the tool calls.

**3. Structured concierge.** Give `concierge_v1` an `output_type` with `flight_id`,
`hotel_id` and `total_inr`. What happens to the chatty answers?

**4. Persist memory.** Use `SQLiteSession("asha", "trips.db")`, restart the kernel,
and ask "what was my plan?"

**5. Find the slow step.** Run the Part 12 cell and open its trace. Which span took
longest, and was it the model or a tool?